In [24]:
import pandas as pd 
from xgboost import XGBClassifier
from pathlib import Path

In [25]:
PROJECT_ROOT = Path.cwd().parent

MODEL_DIR = PROJECT_ROOT
MODELPATH_DIR = PROJECT_ROOT / "models"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

In [26]:
PROJECT_ROOT

WindowsPath('c:/Users/AYAN SONI/Documents/All Projects/nullthreat/ml_model')

In [27]:
model = XGBClassifier()

model.load_model(
    MODELPATH_DIR / "xgboost_baseline_14_features.json"
)

In [28]:
X_train_processed = pd.read_csv(
    ARTIFACTS_DIR / "X_train_processed.csv"
)

X_val_processed = pd.read_csv(
    ARTIFACTS_DIR / "X_val_processed.csv"
)

X_test_processed = pd.read_csv(
    ARTIFACTS_DIR / "X_test_processed.csv"
)

In [29]:
y_train = pd.read_csv(
    ARTIFACTS_DIR / "y_train.csv"
).squeeze("columns")

y_val = pd.read_csv(
    ARTIFACTS_DIR / "y_val.csv"
).squeeze("columns")

y_test = pd.read_csv(
    ARTIFACTS_DIR / "y_test.csv"
).squeeze("columns")

In [30]:
print("=" * 60)
print("Loaded Processed Dataset")
print("=" * 60)

print("\nTraining:")
print(X_train_processed.shape)
print(y_train.shape)

print("\nValidation:")
print(X_val_processed.shape)
print(y_val.shape)

print("\nTesting:")
print(X_test_processed.shape)
print(y_test.shape)

Loaded Processed Dataset

Training:
(188738, 14)
(188738,)

Validation:
(23005, 14)
(23005,)

Testing:
(23627, 14)
(23627,)


In [31]:
print("\nModel features:", model.n_features_in_)
print("Dataset features:", X_train_processed.shape[1])

print("\nModel feature names:")
print(model.feature_names_in_)

print("\nDataset feature names:")
print(X_train_processed.columns.tolist())


Model features: 14
Dataset features: 14

Model feature names:
['URLSimilarityIndex' 'CharContinuationRate' 'LineOfCode' 'HasFavicon'
 'Robots' 'HasDescription' 'HasSocialNet' 'HasHiddenFields'
 'HasCopyrightInfo' 'NoOfImage' 'NoOfCSS' 'NoOfJS' 'NoOfSelfRef'
 'NoOfExternalRef']

Dataset feature names:
['URLSimilarityIndex', 'CharContinuationRate', 'LineOfCode', 'HasFavicon', 'Robots', 'HasDescription', 'HasSocialNet', 'HasHiddenFields', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfExternalRef']


In [32]:
import pandas as pd


# ---------------------------------------------------------
# XGBoost Feature Importance
# ---------------------------------------------------------

feature_importance = pd.DataFrame({
    "feature": X_train_processed.columns,
    "importance": model.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values(
        by="importance",
        ascending=False
    )
    .reset_index(drop=True)
)

feature_importance["rank"] = (
    feature_importance.index + 1
)

feature_importance = feature_importance[
    ["rank", "feature", "importance"]
]


print("=" * 60)
print("XGBoost Feature Importance")
print("=" * 60)

print(
    feature_importance.to_string(
        index=False
    )
)

XGBoost Feature Importance
 rank              feature  importance
    1   URLSimilarityIndex    0.846806
    2      NoOfExternalRef    0.097067
    3          NoOfSelfRef    0.036475
    4           LineOfCode    0.007067
    5       HasDescription    0.003248
    6         HasSocialNet    0.002854
    7 CharContinuationRate    0.002262
    8            NoOfImage    0.001961
    9     HasCopyrightInfo    0.000874
   10              NoOfCSS    0.000729
   11               NoOfJS    0.000417
   12           HasFavicon    0.000107
   13               Robots    0.000090
   14      HasHiddenFields    0.000046


In [33]:
# ---------------------------------------------------------
# XGBoost Gain Importance
# ---------------------------------------------------------

booster = model.get_booster()

gain_scores = booster.get_score(
    importance_type="gain"
)

gain_importance = pd.DataFrame(
    gain_scores.items(),
    columns=[
        "feature",
        "gain"
    ]
)

gain_importance = (
    gain_importance
    .sort_values(
        by="gain",
        ascending=False
    )
    .reset_index(drop=True)
)

gain_importance["rank"] = (
    gain_importance.index + 1
)

gain_importance = gain_importance[
    ["rank", "feature", "gain"]
]


print("=" * 60)
print("XGBoost Gain Importance")
print("=" * 60)

print(
    gain_importance.to_string(
        index=False
    )
)

XGBoost Gain Importance
 rank              feature        gain
    1   URLSimilarityIndex 7203.994629
    2      NoOfExternalRef  825.770325
    3          NoOfSelfRef  310.298676
    4           LineOfCode   60.116634
    5       HasDescription   27.633665
    6         HasSocialNet   24.280043
    7 CharContinuationRate   19.239529
    8            NoOfImage   16.684000
    9     HasCopyrightInfo    7.431576
   10              NoOfCSS    6.202854
   11               NoOfJS    3.543730
   12           HasFavicon    0.906562
   13               Robots    0.763359
   14      HasHiddenFields    0.387143


In [34]:
# ---------------------------------------------------------
# Combine Importance Measures
# ---------------------------------------------------------

analysis = feature_importance.merge(
    gain_importance,
    on="feature",
    how="left"
)


analysis = analysis[
    [
        "feature",
        "importance",
        "gain"
    ]
]


print("=" * 60)
print("Combined XGBoost Feature Analysis")
print("=" * 60)

print(
    analysis.to_string(
        index=False
    )
)

Combined XGBoost Feature Analysis
             feature  importance        gain
  URLSimilarityIndex    0.846806 7203.994629
     NoOfExternalRef    0.097067  825.770325
         NoOfSelfRef    0.036475  310.298676
          LineOfCode    0.007067   60.116634
      HasDescription    0.003248   27.633665
        HasSocialNet    0.002854   24.280043
CharContinuationRate    0.002262   19.239529
           NoOfImage    0.001961   16.684000
    HasCopyrightInfo    0.000874    7.431576
             NoOfCSS    0.000729    6.202854
              NoOfJS    0.000417    3.543730
          HasFavicon    0.000107    0.906562
              Robots    0.000090    0.763359
     HasHiddenFields    0.000046    0.387143


In [35]:
print("\nSame feature names and order:")

print(
    list(model.feature_names_in_)
    == X_train_processed.columns.tolist()
)


Same feature names and order:
True


In [36]:
print(
    X_train_processed.assign(
        label=y_train.values
    )
    .groupby("label")["URLSimilarityIndex"]
    .agg(
        ["count", "min", "max", "mean", "median", "std"]
    )
)

        count         min    max        mean      median        std
label                                                              
0       80811    0.155574  100.0   49.448674   51.309008  22.689871
1      107927  100.000000  100.0  100.000000  100.000000   0.000000


In [37]:
print(
    X_train_processed["URLSimilarityIndex"].describe()
)

count    188738.000000
mean         78.355693
std          29.087909
min           0.155574
25%          56.963686
50%         100.000000
75%         100.000000
max         100.000000
Name: URLSimilarityIndex, dtype: float64


In [39]:
# Trial analysis 

TRIALS_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "xgboost_trials.csv"
)

trials = pd.read_csv(TRIALS_PATH)

print("=" * 60)
print("Optuna Trial Analysis")
print("=" * 60)

print("\nShape:")
print(trials.shape)

print("\nColumns:")
print(trials.columns.tolist())


Optuna Trial Analysis

Shape:
(50, 15)

Columns:
['number', 'value', 'datetime_start', 'datetime_complete', 'duration', 'params_colsample_bytree', 'params_gamma', 'params_learning_rate', 'params_max_depth', 'params_min_child_weight', 'params_n_estimators', 'params_reg_alpha', 'params_reg_lambda', 'params_subsample', 'state']


In [40]:
print("=" * 60)
print("Trial State Analysis")
print("=" * 60)

print(trials["state"].value_counts())

print("\nCompleted trials:")
print((trials["state"] == "COMPLETE").sum())

print("\nFailed trials:")
print((trials["state"] == "FAIL").sum())

Trial State Analysis
state
COMPLETE    50
Name: count, dtype: int64

Completed trials:
50

Failed trials:
0


In [41]:
print("\n" + "=" * 60)
print("F1 Score Distribution")
print("=" * 60)

print(trials["value"].describe())


F1 Score Distribution
count    50.000000
mean      0.996304
std       0.000398
min       0.994991
25%       0.996207
50%       0.996410
75%       0.996566
max       0.996870
Name: value, dtype: float64


In [42]:
top_trials = (
    trials[
        trials["state"] == "COMPLETE"
    ]
    .sort_values(
        "value",
        ascending=False
    )
    .head(10)
)

print("=" * 60)
print("TOP 10 OPTUNA TRIALS")
print("=" * 60)

print(
    top_trials[
        [
            "number",
            "value",
            "params_n_estimators",
            "params_max_depth",
            "params_learning_rate",
            "params_min_child_weight",
            "params_gamma",
            "params_subsample",
            "params_colsample_bytree",
            "params_reg_alpha",
            "params_reg_lambda"
        ]
    ].to_string(index=False)
)

TOP 10 OPTUNA TRIALS
 number    value  params_n_estimators  params_max_depth  params_learning_rate  params_min_child_weight  params_gamma  params_subsample  params_colsample_bytree  params_reg_alpha  params_reg_lambda
     11 0.996870                  273                10              0.170030                        1      0.156896          0.715734                 0.617443          0.001180           0.002114
     12 0.996723                  228                10              0.027983                        1      0.100052          0.729925                 0.713427          0.000186           0.020406
     47 0.996723                  219                 9              0.135955                        1      0.494383          0.827609                 0.735407          0.000493           1.749774
     43 0.996723                  297                 9              0.149783                        1      0.039986          0.746799                 0.734362          2.460930           0.0

In [43]:
print("\n" + "=" * 60)
print("BASELINE VS OPTUNA")
print("=" * 60)

baseline_f1 = 0.9986293123408196
optuna_f1 = trials["value"].max()

print(f"Baseline F1 : {baseline_f1:.6f}")
print(f"Optuna F1   : {optuna_f1:.6f}")
print(f"Difference  : {optuna_f1 - baseline_f1:.6f}")


BASELINE VS OPTUNA
Baseline F1 : 0.998629
Optuna F1   : 0.996870
Difference  : -0.001759
